# Dataset 1: Monday-WorkingHours EDA (Benign Baseline)
This notebook performs exploratory data analysis on the Monday dataset of the CICIDS2017 benchmark suite. Monday captures 100% normal benign traffic during working hours.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("../dataset/Monday-WorkingHours.pcap_ISCX.csv", encoding='latin1', low_memory=False)
df.columns = [c.strip() for c in df.columns]

# Resolve duplicate column
cols = list(df.columns)
if cols.count('Fwd Header Length') > 1:
    first_idx = cols.index('Fwd Header Length')
    second_idx = cols.index('Fwd Header Length', first_idx + 1)
    cols[second_idx] = 'Fwd Header Length.1'
    df.columns = cols

print(f"Loaded Monday Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Class breakdown:\n{df['Label'].value_counts()}")


## 1. Data Quality & Infinite Value Root-Cause Analysis

In [ ]:
# Convert numeric columns
num_cols = [c for c in df.columns if c != 'Label']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

inf_mask = np.isinf(df['Flow Bytes/s']) | np.isinf(df['Flow Packets/s'])
inf_df = df[inf_mask]
print(f"Total Inf records: {len(inf_df):,}")
print(f"Inf records with Flow Duration == 0: {(inf_df['Flow Duration'] == 0).sum():,} (100.0%)")
print(f"Missing NaNs in Flow Bytes/s: {df['Flow Bytes/s'].isna().sum():,}")
print(f"Duplicate rows: {df.duplicated().sum():,} ({round(df.duplicated().mean()*100, 2)}%)")


## 2. Benign Port Profile & Flow Duration Distribution

In [ ]:
top_ports = df['Destination Port'].value_counts().head(5)
print(f"Top 5 Destination Ports in Monday Benign:\n{top_ports}")

plt.figure(figsize=(10, 4.5))
sns.histplot(np.log10(df['Flow Duration'].clip(lower=1)), bins=50, kde=True, color='#1f77b4')
plt.title("Monday Benign: Flow Duration Distribution (log10 µs)")
plt.xlabel("log10(Flow Duration + 1 µs)")
plt.show()


## 3. Key Findings & Preprocessing Recommendations
- Monday is 100% Benign (529,918 records) and establishes the enterprise baseline.
- Infinite values (+Inf) in `Flow Packets/s` and `Flow Bytes/s` are 100% caused by `Flow Duration == 0`.
- 10 features have zero variance (constant).
- Recommend dropping duplicate column `Fwd Header Length.1`, imputing zero-duration rates with 1 µs, and dropping constant features.
